# Section 03: 微调掩码语言模型（MLM）核心总结

## 任务定义
**Masked Language Model (MLM) 继续预训练** = 用领域数据对已有预训练模型进行「二次预训练」，使其更适应特定领域，再做下游任务微调。

**为什么需要这一步？**
- BERT/DistilBERT 在通用语料上预训练，对专业领域词汇（如电影评论口语、医学术语）理解较弱
- 在下游任务微调前，先用领域数据做 MLM 二次预训练，可提升最终效果

## 本节任务
用 **IMDB 电影评论**数据对 `distilbert-base-uncased` 做 MLM 继续预训练

## 完整流程
```
IMDB 数据集（纯文本，无需标签）
    ↓ 分词（不截断，保留 word_ids）
    ↓ group_texts（拼接 + 切块，消除截断浪费）
    ↓ DataCollatorForLanguageModeling（随机 mask 15% token）
    ↓ 或 whole_word_masking（整词 mask）
    ↓ AutoModelForMaskedLM 训练
    ↓ 评估指标：Perplexity（困惑度）
```

---
## 第一步：MLM 预训练目标回顾

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
import torch

model_checkpoint = "distilbert/distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# DistilBERT 是 BERT 的蒸馏版：参数量减少 40%，速度快 60%，性能保留 97%
print(f"DistilBERT 参数量: {model.num_parameters()/1e6:.0f}M")
print(f"BERT 参数量: 110M")

# 演示：MLM 如何工作——预测被 [MASK] 遮盖的 token
text = "This is a great [MASK]."
inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits

# 找到 [MASK] 的位置，取出对应 logit，预测 top-5
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(text.replace("[MASK]", tokenizer.decode([token])))

---
## 第二步：数据准备 — group_texts（核心技巧）

### 问题
MLM 不需要标签，可以用**全部文本**训练。
但直接 truncation 会丢弃超出 512 的部分（评论文本通常很长）。

### 解决方案：拼接后切块
```
评论1 (363 tokens) + 评论2 (304 tokens) + 评论3 (133 tokens) = 800 tokens
    ↓ 按 chunk_size=128 切分
块1[0:128]  块2[128:256]  块3[256:384]  块4[384:512]  块5[512:640]  块6[640:768]  (丢弃最后32)
```
优势：**零浪费**，每个 token 都参与训练

In [ ]:
from datasets import load_dataset

imdb_dataset = load_dataset("imdb")

# 第一步：分词（不截断！保留 word_ids 用于 Whole Word Masking）
def tokenize_function(examples):
    result = tokenizer(examples["text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result

tokenized_datasets = imdb_dataset.map(
    tokenize_function, batched=True,
    remove_columns=["text", "label"]  # MLM 不需要标签
)

# 第二步：拼接 + 切块
chunk_size = 128

def group_texts(examples):
    # 将批次内所有样本的 token 拼接成一个长序列
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated[list(examples.keys())[0]])
    
    # 丢弃最后不足 chunk_size 的部分，避免 padding
    total_length = (total_length // chunk_size) * chunk_size
    
    # 切分成固定长度的块
    result = {
        k: [t[i:i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated.items()
    }
    # MLM 的 labels 就是原始 input_ids（被 mask 后由 data_collator 处理）
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_datasets.map(group_texts, batched=True)
print(f"原始训练样本数: {len(imdb_dataset['train'])}")
print(f"切块后训练样本数: {len(lm_datasets['train'])}")
# 25000条评论 → 61291个128-token块

---
## 第三步：两种 Masking 策略

### 策略A：Token 级别随机 Mask（标准 MLM）
随机选 15% 的 **token** 进行 mask，subword 可能被单独 mask。

### 策略B：Whole Word Masking（整词 Mask）
随机选 20% 的**词**（word），该词的所有 subword 一起被 mask。
更难的预训练任务 → 通常效果更好。

In [ ]:
from transformers import DataCollatorForLanguageModeling

# ── 策略A：标准 Token-level Masking ─────────────────────────────
# mlm_probability=0.15 表示随机 mask 15% 的 token
# DataCollatorForLanguageModeling 会自动：
#   - 随机选 15% token
#   - 80% 替换为 [MASK]
#   - 10% 替换为随机 token
#   - 10% 保持不变
#   - 只对这些位置计算 loss（其余位置 label=-100）
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

print("策略A 示例（token级别随机mask）:")
samples = [lm_datasets["train"][i] for i in range(2)]
for s in samples:
    s.pop("word_ids", None)
for chunk in data_collator(samples)["input_ids"]:
    print(tokenizer.decode(chunk)[:100], "...")

In [ ]:
import collections
import numpy as np
from transformers import default_data_collator

wwm_probability = 0.2

def whole_word_masking_data_collator(features):
    """
    整词 Masking：利用 word_ids 确保同一个词的所有 subword 同时被 mask。
    """
    for feature in features:
        word_ids = feature.pop("word_ids")

        # 建立 word_index → token_indices 的映射
        # 例如: {0: [1], 1: [2,3], 2: [4]} 表示词1被拆成了2个subword
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                mapping[current_word_index].append(idx)

        # 以词为单位随机决定是否 mask（二项分布采样）
        mask = np.random.binomial(1, wwm_probability, (len(mapping),))

        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)  # 默认全部忽略

        for word_id in np.where(mask)[0]:
            # 该词被选中：将其所有 subword token 都替换为 [MASK]
            for idx in mapping[word_id.item()]:
                new_labels[idx] = labels[idx]  # 恢复该位置的 label（用于计算 loss）
                input_ids[idx] = tokenizer.mask_token_id

        feature["labels"] = new_labels

    return default_data_collator(features)

---
## 第四步：训练与评估指标 — Perplexity（困惑度）

**Perplexity = exp(loss)**
- 直觉：模型对下一个 token 有多「困惑」，值越低越好
- Perplexity=10 意味着模型平均在 10 个候选中才能找到正确答案
- 二次预训练后 perplexity 应显著下降（模型更适应领域语言）

In [ ]:
import math
from transformers import TrainingArguments, Trainer

# 为节省时间，只用 10000 个样本
train_size = 10_000
test_size = int(0.1 * train_size)
downsampled_dataset = lm_datasets["train"].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)

batch_size = 64
logging_steps = len(downsampled_dataset["train"]) // batch_size

training_args = TrainingArguments(
    output_dir="distilbert-base-uncased-finetuned-imdb",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=data_collator,  # 用 token-level mask
    processing_class=tokenizer,
)

# 训练前评估基准 perplexity
eval_results = trainer.evaluate()
print(f"训练前 Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

# trainer.train()
# eval_results = trainer.evaluate()
# print(f"训练后 Perplexity: {math.exp(eval_results['eval_loss']):.2f}")
# 实际结果：11.39 → 11.46（过拟合，因为 IMDB 与通用语料差异不大）

---
## 第五步：推理验证

对比预训练模型和微调后模型对电影领域文本的预测差异。

In [ ]:
from transformers import pipeline

# 微调前（通用模型）
mask_filler_base = pipeline("fill-mask", model="distilbert/distilbert-base-uncased")
text = "This is a great [MASK]."
print("通用模型预测:")
for pred in mask_filler_base(text):
    print(f"  {pred['sequence']} (score: {pred['score']:.4f})")

# 微调后（IMDB领域模型）
mask_filler_imdb = pipeline("fill-mask", model="goosmanlei/distilbert-base-uncased-finetuned-imdb-accelerate")
print("\nIMDB微调后模型预测:")
for pred in mask_filler_imdb(text):
    print(f"  {pred['sequence']} (score: {pred['score']:.4f})")
# 微调后：film/movie 的概率更高，更符合电影评论语境

---
## 总结

### 核心知识点

| 概念 | 说明 |
|------|------|
| MLM 二次预训练 | 在领域数据上继续 MLM 训练，提升领域适应性（无需标签）|
| group_texts | 拼接文本后切块，避免截断浪费，充分利用所有 token |
| Token Masking | 随机 mask 15% token，其中 80%→[MASK], 10%→随机, 10%→不变 |
| Whole Word Masking | 以词为单位 mask，更难的任务，通常效果更好 |
| Perplexity | 评估语言模型质量：exp(loss)，越低越好 |
| labels 设置 | labels = input_ids 的副本；被 mask 位置保留真实 id，其余位置为 -100 |

### MLM vs CLM
```
MLM（BERT类）：随机 mask token，双向注意力，适合理解任务
CLM（GPT类）：预测下一个 token，单向注意力，适合生成任务
```

### DataCollatorForLanguageModeling 的两种模式
```python
# MLM 模式（默认）
DataCollatorForLanguageModeling(tokenizer, mlm=True, mlm_probability=0.15)

# CLM 模式（section-06 使用）
DataCollatorForLanguageModeling(tokenizer, mlm=False)  # labels = input_ids 右移一位
```